# Clothing segmentation with the Hugging Face Inference API

This notebook sends a single image to the pre-trained
[SegFormer B3 Clothes](https://huggingface.co/sayeed99/segformer_b3_clothes) model
through the Hugging Face Inference API, and turns its response into a segmentation mask.

The API returns one binary mask per detected class, encoded in base64.
These masks are decoded and merged into a single mask where each pixel holds
the ID of its class, the same format as the reference masks of the dataset.

In [ ]:
%load_ext autoreload
%autoreload 2

## Configuration

Imports, and the paths used in this notebook:

* `IMAGE_PATH`: the image to segment. Change it to try another image of the dataset.
* `MASK_PATH`: the reference mask of that image, annotated by hand.
* `PREDICTED_MASK_PATH`: where the mask predicted by the model is saved, in `data/output/`.

`load_dotenv()` reads the `.env` file at the root of the project, which holds the Hugging Face token (`HF_TOKEN`).

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv
from oc_p02_fashion_segmentation.hf_api import get_hf_token, segment_image
from oc_p02_fashion_segmentation.masks import create_masks, save_mask
from oc_p02_fashion_segmentation.visualization import display_comparison, display_overlay

DATA_DIR = Path("../data/input")
IMAGE_NUMBER = 2
IMAGE_PATH = DATA_DIR / "IMG" / f"image_{IMAGE_NUMBER}.png"
MASK_PATH = DATA_DIR / "Mask" / f"mask_{IMAGE_NUMBER}.png"
OUTPUT_DIR = Path("../data/output")
# Named like the dataset masks: image_0.png -> mask_0.png
PREDICTED_MASK_PATH = OUTPUT_DIR / IMAGE_PATH.name.replace("image_", "mask_")

load_dotenv()

## Query the API

The image is loaded to get its size: the masks returned by the API are resized to it in the next step.

`segment_image` sends the image to the model and returns a list with one entry per detected class: its `label` (the class name) and its `mask` (a binary PNG encoded in base64). It raises an `HFAPIError` if the token is missing or invalid, the request times out, or the API returns an error.

In [ ]:
image = Image.open(IMAGE_PATH)
image_array = np.array(image)
image_width, image_height = image.size

result = segment_image(get_hf_token(), IMAGE_PATH)

## Save the predicted mask and verify it

`create_masks` merges the masks returned by the API into a single mask, where each pixel holds the ID of its class, resized to the image.

The mask is saved as PNG, a lossless format, so every pixel keeps its exact class ID. With JPEG, compression would alter pixels near class edges and turn them into other classes.

The saved mask is then loaded and checked against the original mask to verify that it was saved correctly.

In [ ]:
mask_array = create_masks(result, image_width, image_height)
save_mask(mask_array, PREDICTED_MASK_PATH)
print(f"Predicted mask saved to {PREDICTED_MASK_PATH}")

# Load the saved mask and verify it
saved_mask = Image.open(PREDICTED_MASK_PATH)
saved_mask_array = np.array(saved_mask)

# Check if the saved mask matches the predicted mask
if np.array_equal(mask_array, saved_mask_array):
    print("Mask verification successful: The saved mask matches the predicted mask.")
else:
    print("Mask verification failed: The saved mask does not match the predicted mask.")

## Visualize the result

Overlay the predicted mask on the original image, with one color per class.

In [ ]:
display_overlay(image_array, mask_array)

## Compare with the reference mask

Four panels, from left to right:

* the original image,
* the reference mask, annotated by hand, overlaid on the image,
* the mask predicted by the model, overlaid on the image,
* the pixels where the two masks disagree, in a single vivid colour. The title gives their share of the image.

In [ ]:
reference_mask_array = np.array(Image.open(MASK_PATH))
display_comparison(image_array, reference_mask_array, mask_array)